# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the [FAIR² dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. All dataset entities (record sets, fields, columns) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset is defined by a Croissant schema, accessible via this URL:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. We'll print a description from the metadata for orientation.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

We will list all available record sets (by their `@id`s), their respective fields and columns, and provide a brief schema view using Croissant discovery. Use the `@id` values for precise reference and selection.

In [ ]:
# List available record sets, fields, and columns, all referenced by their `@id`

def list_recordsets_and_fields(ds):
    record_sets = ds.metadata.record_sets
    print(f"\nFound {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"- Record set: {rs['@id']} (name: {rs.get('name','-')})")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("    Fields:")
        for f in fields:
            print(f"      - {f['@id']} (name: {f.get('name','-')}, dataType: {f.get('dataType','-')})")
            if 'column' in f:
                columns = f['column']
                if isinstance(columns, dict):
                    columns = [columns]
                print("            Columns:")
                for c in columns:
                    print(f"               - {c['@id']} (name: {c.get('name','-')})")
    return record_sets

recordsets_objs = list_recordsets_and_fields(dataset)

### Example: Peek at records from a specific record set

Let's preview records from a record set using its `@id`.

In [ ]:
# Get all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Available record sets by @id:")
for rsid in record_set_ids:
    print("  ", rsid)

# Pick the first record set for demonstration:
if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print(f"\nSample records from record set @id: {sample_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(rec)
        if i > 2:
            print("...")
            break

## 3. Data Extraction

Now, we load all data from each record set into a `pandas.DataFrame` for subsequent analysis. All extraction uses record set, field, and column `@id` references.

In [ ]:
# Reuse list of record set ids from earlier
dataframes = {}
print('Record sets to extract:', record_set_ids)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for record set '@id': {record_set_id}")
    print(df.columns.tolist())
    print(df.head(3))

# We'll use the first record set as the main example for further analysis:
main_record_set_id = record_set_ids[0]
df_main = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)

Let's:
- Filter records based on a numeric field (`@id` reference),
- Normalize that numeric field (standardization),
- Optionally group by a categorical field if present.

Inspect column `@id`s and types to select appropriate fields.

In [ ]:
# Inspect column names and dtypes
print("Columns and dtypes in main DataFrame:")
print(df_main.dtypes)

# Select a numeric field by @id -- update this to a field present in your dataset
# For illustration, let's pick the first numeric-looking column (e.g. 'age' or interval type variable)
import numpy as np
numeric_columns = [col for col in df_main.columns if pd.api.types.is_numeric_dtype(df_main[col])]
if not numeric_columns:
    # Try to convert some plausible columns
    for col in df_main.columns:
        # Try to coerce to float
        try:
            df_main[col] = pd.to_numeric(df_main[col], errors='coerce')
        except Exception:
            continue
    numeric_columns = [col for col in df_main.columns if pd.api.types.is_numeric_dtype(df_main[col])]

if numeric_columns:
    numeric_field_id = numeric_columns[0]
    print(f"Using numeric field '@id': {numeric_field_id}")
else:
    raise ValueError("No numeric field found for EDA.")

# Filtering: Remove rows with missing values for this numeric field, then filter by a threshold
threshold = df_main[numeric_field_id].mean()
filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
print(f"\nFiltered records in '{main_record_set_id}' with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} rows")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping: Try by a likely categorical field (with a small number of unique values)
categorical_fields = [col for col in df_main.columns if df_main[col].nunique() < 10 and col != numeric_field_id]
group_field_id = categorical_fields[0] if categorical_fields else None
if group_field_id:
    print(f"\nGrouping by field '@id': {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(grouped_df)
else:
    print("\nNo suitable categorical field found for grouping.")

## 5. Visualization

We can visualize the distribution of our selected numeric field and the effect of grouping by a categorical field. (You can extend this to any other analysis as appropriate for the dataset.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df_main[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If we have a grouping field, show boxplot
if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df_main, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

Using the `mlcroissant` library we have:
- Discovered and referenced dataset entities by their `@id` for robust, standards-based data handling.
- Loaded and previewed all available record sets and fields in the FAIR² dataset describing second primary colorectal cancers in cancer survivors.
- Performed filtering, normalization, and grouping on one numeric field by `@id`.
- Visualized data distributions to aid in further analysis.

**Next steps:** Consider advanced analyses such as feature selection, supervised modeling, or cross-record set joins, always tracking entities by their `@id` for provenance and reproducibility.